In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab-mistral/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Rate limits, retries, breakers and admission — the 429 and the queue that never says no

**What you'll learn.** What a 429 means on Mistral's API, why a vLLM fleet never sends one (it queues and slows
down instead), why you smooth your own traffic, why retries need jitter, what a circuit breaker buys, and how
admission control with degrade levels breaks the overload feedback loop in both of its forms. Code:
`scalelab/model.py` (the two backends), `scalelab/resilience.py` and `scalelab/admission.py`.

> **In a design review.** *"On the API a 429 is contention on a per-model limit the whole workspace shares. On your own
> GPUs there is no 429 — the queue grows and every user gets slower, with zero errors on the dashboard. Either way
> I smooth my traffic, retry with jitter inside the turn's deadline, break the circuit to a sibling, and — before
> any of that — admit only as many turns as the tokens can carry."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 160)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The API's limit

A sliding 60-second window of tokens *and* a requests-per-second cap, per model. Above 75 % utilisation latency inflates; at the limit the pool answers 429 with a retry hint. (Mistral documents no `Retry-After` header — the SDK honours one if present; plan for backoff without it.)

In [ ]:
from scalelab.model import SharedPool, ServerPool, HostedBackend, HybridBackend
from scalelab.resilience import RateLimited, TokenBucket, backoff, CircuitBreaker, call_with_retries
pool = SharedPool(tpm=100_000, rps=50)
admitted = 0
try:
    while True:
        pool.admit(5_000); admitted += 1
except RateLimited as e:
    print(f"{admitted} calls of 5k tokens admitted, then 429 with retry_after={e.retry_after:.1f}s; utilisation {pool.utilisation:.0%}")
burst = SharedPool(tpm=10_000_000, rps=5); n = 0
try:
    while True:
        burst.admit(100); n += 1
except RateLimited:
    print(f"a 5-RPS limit: {n} requests in one second, then 429 regardless of tokens")

## 2. The fleet: nobody is refused, everybody is slower

One Ministral 14B replica on an H100. Send *k* calls at once: no 429, no error — the batch grows, the time per
token grows with it, and every call in the batch pays. Aggregate throughput rises with the batch; per-call latency
rises faster once past the batch where the target TPOT holds (dashed line).

In [ ]:
from scalelab.serving import replica
rows = []
for k in (1, 8, 16, 24, 32, 48, 64, 96):
    CLOCK.reset(0.02)
    srv = ServerPool(replica("ministral-14b", "h100"), replicas=1, context_tokens=5_200, shared_prefix_tokens=3_000)
    rng = random.Random(0); t0 = CLOCK.now()
    lat = await asyncio.gather(*(srv.serve(5_000, 3_000, 300, rng) for _ in range(k)))
    rows.append({"concurrent calls": k, "p50 call latency (s)": np.median([l for l, _ in lat]), "tokens/s served": k * 300 / (CLOCK.now() - t0), "errors": srv.rejected})
df = pd.DataFrame(rows).set_index("concurrent calls").round(2); print(df)
ax = df[["p50 call latency (s)"]].plot(figsize=(6.5, 3.4), marker="o", title="One replica: latency vs concurrency (300-token answers)")
ax.axvline(srv.target_batch, ls="--", c="grey"); ax.set_ylabel("seconds"); ax.grid(alpha=.3); plt.tight_layout()

## 3. Smooth your own traffic

A token bucket refills at the rate you chose and tolerates a burst worth `capacity`. Thirty calls arriving at once leave at a steady pace instead of hitting the limit together. On the fleet the same bucket keeps the batch near the target instead of letting a burst push every replica past it.

In [ ]:
async def burst(bucket):
    t = []
    for _ in range(30):
        await bucket.acquire(5_000); t.append(CLOCK.now())
    return np.array(t) - t[0]
CLOCK.reset(0.02)
smooth = await burst(TokenBucket(rate=50_000 / 60, capacity=50_000 / 60 * 6))    # 50k TPM, 6-second burst
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.eventplot([np.zeros(30), smooth], lineoffsets=[1, 0], colors=["tab:red", "tab:blue"])
ax.set_yticks([1, 0]); ax.set_yticklabels(["unsmoothed burst", "through the bucket"]); ax.set_xlabel("virtual seconds"); ax.set_title("30 calls of 5k tokens")
plt.tight_layout()
print(f"the burst is spread over {smooth[-1]:.1f} s at the sustained rate of the bucket")

## 4. Jitter

When the limit throttles everyone at once, everyone retries at once — unless the backoff is jittered. Two hundred clients, each retrying until a pool with room for 40 calls per second admits them.

In [ ]:
async def retry_storm(jitter, clients=200, rate=40):
    CLOCK.reset(0.02)
    pool = SharedPool(tpm=rate * 60 * 1000)          # 1k tokens per call, `rate` calls/s
    done, extra_429 = [], 0
    async def client(i):
        nonlocal extra_429
        rng = random.Random(i)
        for attempt in range(1, 8):
            try:
                pool.admit(1000); done.append(CLOCK.now()); return
            except RateLimited:
                if attempt > 1: extra_429 += 1
                await CLOCK.sleep(backoff(attempt, jitter=jitter, rng=rng))
    await asyncio.gather(*(client(i) for i in range(clients)))
    return np.array(done), extra_429

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, jitter in zip(axes, (False, True)):
    t, extra = await retry_storm(jitter)
    ax.hist(t, bins=40); ax.set_title(f"{'full jitter' if jitter else 'no jitter'}: {len(t)} served by {t.max():.1f}s, {extra} secondary 429s"); ax.set_xlabel("virtual seconds")
plt.tight_layout()

## 5. The circuit breaker

After enough failures in a window it opens (fail fast for a cooldown), then half-opens for a single probe. A breaker per model lets the gateway move to a sibling model whose limit is not contended — or, on a fleet, from the saturated pool to the API spill-over.

In [ ]:
CLOCK.reset(0.02)
cb = CircuitBreaker(threshold=3, min_calls=4, ratio=0.5, cooldown=5.0)
for ok in (True, False, False, False):
    cb.record(ok); print(f"record({ok}) -> {cb.state}")
await CLOCK.sleep(5.1); print("after cooldown ->", cb.state)
cb.record(True); print("probe succeeded ->", cb.state)

## 6. Admission control: the brake

Signals → level. On the API the signals are the in-flight cap and the share of calls pushed back (429). On the fleet
the pushback signal is silent, so the controller also reads *saturation*: running + waiting against the batch the
latency target allows (the Kubernetes inference gateway's saturation detector uses the same idea: queue depth ≥ 5
or KV usage ≥ 0.8). Levels 1–2 buy capacity by giving something up (cheaper model or shorter answers, no writes);
level 3 sheds with a `Retry-After`. Levels 1–2 are held for a dwell time so the system does not flap.

In [ ]:
from scalelab.admission import AdmissionController, AdmissionConfig
CLOCK.reset(0.02)
ac = AdmissionController(AdmissionConfig(max_inflight=10, dwell_s=0))
rows = []
for inflight, ratio, sat in [(0, 0.0, 0.0), (8, 0.0, 0.0), (0, 0.06, 0.0), (0, 0.2, 0.0), (0, 0.0, 0.85), (0, 0.0, 1.3), (10, 0.0, 0.0)]:
    ac.inflight = inflight; ac._recent.clear()
    for _ in range(20): ac.note_model_call(rate_limited=random.random() < ratio)
    rows.append({"inflight": inflight, "pushback ratio": ratio, "fleet saturation": sat, "level": ac.compute_level(saturation=sat)})
pd.DataFrame(rows)

## Your turn — solutions

#### (a) Backoff with full jitter

min(cap, base·2^(attempt−1)), then a uniform draw between 0 and that.

In [ ]:
def my_backoff(attempt, base=0.5, cap=8.0, rng=random):
    return rng.uniform(0, min(cap, base * 2 ** (attempt - 1)))

In [ ]:
def _a():
    rng = random.Random(3)
    draws = [my_backoff(a, rng=rng) for a in range(1, 12) for _ in range(50)]
    assert all(0 <= d <= 8.0 for d in draws)
    assert max(my_backoff(3, rng=random.Random(k)) for k in range(200)) <= 2.0
    assert max(my_backoff(6, rng=random.Random(k)) for k in range(200)) > 6.0
check("a: backoff", _a)

#### (b) A token bucket

Implement `try_acquire`: refill by elapsed time × rate (capped), then either take the tokens (return 0) or return the seconds to wait.

In [ ]:
class MiniBucket:
    def __init__(self, rate, capacity):
        self.rate, self.capacity, self.tokens, self.last = rate, capacity, capacity, CLOCK.now()
    def try_acquire(self, n=1.0):
        now = CLOCK.now()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate); self.last = now
        if self.tokens >= n:
            self.tokens -= n; return 0.0
        return (n - self.tokens) / self.rate

In [ ]:
def _b():
    CLOCK.reset(0.02)
    b = MiniBucket(rate=10, capacity=10)
    assert all(b.try_acquire(1) == 0 for _ in range(10))
    w = b.try_acquire(5)
    assert 0.3 < w <= 0.5, w
check("b: token bucket", _b)

#### (c) Degrade level with the fleet's signal

Reproduce the level function without hysteresis: 1 above 80 % of the cap, or ≥ 5 % pushback, or queue age ≥ 10 s, or saturation ≥ 0.8; 2 at ≥ 15 % pushback or saturation ≥ 1.0; 3 at the cap or queue age ≥ 30 s.

In [ ]:
def degrade_level(inflight, cap, pushback_ratio, queue_age_s, saturation=0.0):
    if inflight >= cap or queue_age_s >= 30: return 3
    if pushback_ratio >= 0.15 or saturation >= 1.0: return 2
    if inflight >= 0.8 * cap or pushback_ratio >= 0.05 or queue_age_s >= 10 or saturation >= 0.8: return 1
    return 0

In [ ]:
def _c():
    assert degrade_level(10, 100, 0, 0) == 0
    assert degrade_level(85, 100, 0, 0) == 1 and degrade_level(0, 100, 0.06, 0) == 1 and degrade_level(0, 100, 0, 12) == 1
    assert degrade_level(0, 100, 0, 0, 0.85) == 1 and degrade_level(0, 100, 0, 0, 1.2) == 2
    assert degrade_level(0, 100, 0.2, 0) == 2
    assert degrade_level(100, 100, 0, 0) == 3 and degrade_level(0, 100, 0, 31) == 3
check("c: degrade level", _c)

#### (d) The spill-over decision

Route a model call: 'fleet' while the fleet's saturation is below `spill_at`; otherwise 'api' if the call carries no personal data (the residency policy) and the API's breaker is closed; otherwise 'degrade' (stay on the fleet with shorter answers). Priority callers (≤ 2) always get the fleet.

In [ ]:
def route(saturation, has_pii, api_breaker_closed, priority=5, spill_at=0.9):
    if priority <= 2 or saturation < spill_at:
        return "fleet"
    if not has_pii and api_breaker_closed:
        return "api"
    return "degrade"

In [ ]:
def _d():
    assert route(0.5, True, True) == "fleet"
    assert route(0.95, False, True) == "api"
    assert route(0.95, True, True) == "degrade"           # personal data never leaves the country
    assert route(0.95, False, False) == "degrade"         # the API is broken: stay home
    assert route(1.5, True, True, priority=1) == "fleet"  # priority traffic gets the GPUs
check("d: spill-over", _d)

## Takeaways for the conversation

- 429 = contention on a per-model limit the workspace shares. Smooth within the minute; back off with full jitter; never retry past the deadline. The SDK does not retry for you unless you ask it to — keep it that way so one place decides.
- A fleet never says no: watch batch/TPOT, queue depth and KV usage, and cap the queue (`--max-num-queued-reqs` → 503) so overload becomes a signal instead of a slowdown.
- A breaker per model; a sibling model or the API spill-over as the fallback — gated by the data policy.
- Admission control at the edge with degrade levels: cheaper first, shed last, priorities bypass the cap, hysteresis on the levels.
- The in-flight cap starts from the limit or the fleet (notebook 01) and is tuned from load tests (notebook 04).

## Verify before the conversation

Mistral's documented 429 behaviour and whether any rate-limit headers exist; paid-tier limits in the console; `mistralai` retry defaults (off in 2.10); vLLM's `--max-num-queued-reqs` semantics and the 503 it returns (0.29); the inference gateway's saturation thresholds.